# UK Retail Sales Analysis  
## Notebook 4: Tableau Data Preparation

### Objectives

- Load the historical retail sales dataset and the 12-month forecast.
- Create clear period and performance classifications.
- Reshape the data into a Tableau-friendly format.
- Combine historical and forecast information where appropriate.
- Validate and export the final dashboard datasets.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

## 1. Load the processed datasets

The historical monthly dataset and the forecast created in Notebook 3 are loaded for dashboard preparation.

In [2]:
historical_path = (
    Path("..")
    / "data"
    / "processed"
    / "retail_sales_core_monthly.csv"
)

forecast_path = (
    Path("..")
    / "data"
    / "processed"
    / "retail_sales_volume_forecast.csv"
)

historical = pd.read_csv(
    historical_path,
    parse_dates=["date"]
)

forecast = pd.read_csv(
    forecast_path,
    parse_dates=["date"]
)

print("Historical file found:", historical_path.exists())
print("Forecast file found:", forecast_path.exists())
print("Historical shape:", historical.shape)
print("Forecast shape:", forecast.shape)

Historical file found: True
Forecast file found: True
Historical shape: (356, 6)
Forecast shape: (12, 5)


## 2. Validate the source datasets

Before preparing the Tableau files, the datasets are checked for their column structure, date ranges, missing values and duplicated dates.

In [3]:
print("Historical columns:")
print(historical.columns.tolist())

print("\nForecast columns:")
print(forecast.columns.tolist())

print("\nHistorical date range:")
print(historical["date"].min(), "to", historical["date"].max())

print("\nForecast date range:")
print(forecast["date"].min(), "to", forecast["date"].max())

print("\nHistorical missing values:")
print(historical.isna().sum())

print("\nForecast missing values:")
print(forecast.isna().sum())

print("\nHistorical duplicate dates:", historical["date"].duplicated().sum())
print("Forecast duplicate dates:", forecast["date"].duplicated().sum())

Historical columns:
['date', 'value_yoy_pct', 'value_mom_pct', 'volume_yoy_pct', 'volume_mom_pct', 'implied_deflator_yoy_pct']

Forecast columns:
['date', 'forecast_volume_yoy_pct', 'lower_95_pct', 'upper_95_pct', 'model']

Historical date range:
1997-01-01 00:00:00 to 2026-08-01 00:00:00

Forecast date range:
2026-09-01 00:00:00 to 2027-08-01 00:00:00

Historical missing values:
date                        0
value_yoy_pct               0
value_mom_pct               0
volume_yoy_pct              0
volume_mom_pct              0
implied_deflator_yoy_pct    0
dtype: int64

Forecast missing values:
date                       0
forecast_volume_yoy_pct    0
lower_95_pct               0
upper_95_pct               0
model                      0
dtype: int64

Historical duplicate dates: 0
Forecast duplicate dates: 0


## 3. Prepare the historical dashboard dataset

Additional date fields and business-friendly classifications are created to support filtering, grouping and visualisation in Tableau.

In [4]:
historical_tableau = (
    historical
    .sort_values("date")
    .copy()
)

historical_tableau["year"] = historical_tableau["date"].dt.year
historical_tableau["month_number"] = historical_tableau["date"].dt.month
historical_tableau["month_name"] = historical_tableau["date"].dt.month_name()
historical_tableau["quarter"] = historical_tableau["date"].dt.to_period("Q").astype(str)
historical_tableau["year_month"] = historical_tableau["date"].dt.strftime("%Y-%m")

historical_tableau.head()

,date,value_yoy_pct,value_mom_pct,volume_yoy_pct,volume_mom_pct,implied_deflator_yoy_pct,year,month_number,month_name,quarter,year_month
0,1997-01-01,6.2,1.9,4.5,1.6,2.3,1997,1,January,1997Q1,1997-01
1,1997-02-01,4.6,0.8,3.1,0.8,1.9,1997,2,February,1997Q1,1997-02
2,1997-03-01,5.3,1.2,3.7,1.0,1.4,1997,3,March,1997Q1,1997-03
3,1997-04-01,3.9,0.1,3.0,0.5,1.2,1997,4,April,1997Q2,1997-04
4,1997-05-01,4.4,-0.1,3.2,-0.3,1.1,1997,5,May,1997Q2,1997-05


In [5]:
period_conditions = [
    historical_tableau["year"] <= 2019,
    historical_tableau["year"].between(2020, 2021),
    historical_tableau["year"].between(2022, 2023),
    historical_tableau["year"] >= 2024
]

period_labels = [
    "Pre-pandemic (1997–2019)",
    "COVID-19 disruption (2020–2021)",
    "Cost-of-living crisis (2022–2023)",
    "Recent period (2024–2026)"
]

historical_tableau["economic_period"] = np.select(
    period_conditions,
    period_labels,
    default="Unclassified"
)

historical_tableau["period_order"] = np.select(
    period_conditions,
    [1, 2, 3, 4],
    default=0
)

historical_tableau["volume_growth_status"] = np.select(
    [
        historical_tableau["volume_yoy_pct"] >= 3,
        historical_tableau["volume_yoy_pct"] > 0,
        historical_tableau["volume_yoy_pct"] == 0,
        historical_tableau["volume_yoy_pct"] < 0
    ],
    [
        "Strong growth (3% or more)",
        "Moderate growth (below 3%)",
        "No change",
        "Contraction"
    ],
    default="Unclassified"
)

historical_tableau[
    [
        "date",
        "volume_yoy_pct",
        "economic_period",
        "period_order",
        "volume_growth_status"
    ]
].tail()

,date,volume_yoy_pct,economic_period,period_order,volume_growth_status
351,2026-04-01,0.0,Recent period (2024–2026),4,No change
352,2026-05-01,3.2,Recent period (2024–2026),4,Strong growth (3% or more)
353,2026-06-01,3.4,Recent period (2024–2026),4,Strong growth (3% or more)
354,2026-07-01,1.2,Recent period (2024–2026),4,Moderate growth (below 3%)
355,2026-08-01,2.4,Recent period (2024–2026),4,Moderate growth (below 3%)


## 4. Reshape the historical data for Tableau

The wide dataset is converted into a long format. Each row will represent one date and one retail sales metric, allowing Tableau to compare measures using a single value field.

In [6]:
id_columns = [
    "date",
    "year",
    "month_number",
    "month_name",
    "quarter",
    "year_month",
    "economic_period",
    "period_order",
    "volume_growth_status"
]

metric_columns = [
    "value_yoy_pct",
    "value_mom_pct",
    "volume_yoy_pct",
    "volume_mom_pct",
    "implied_deflator_yoy_pct"
]

historical_long = historical_tableau.melt(
    id_vars=id_columns,
    value_vars=metric_columns,
    var_name="metric_code",
    value_name="percentage"
)

metric_labels = {
    "value_yoy_pct": "Sales value growth",
    "value_mom_pct": "Sales value growth",
    "volume_yoy_pct": "Sales volume growth",
    "volume_mom_pct": "Sales volume growth",
    "implied_deflator_yoy_pct": "Implied deflator growth"
}

comparison_labels = {
    "value_yoy_pct": "Year-on-year",
    "value_mom_pct": "Month-on-month",
    "volume_yoy_pct": "Year-on-year",
    "volume_mom_pct": "Month-on-month",
    "implied_deflator_yoy_pct": "Year-on-year"
}

historical_long["metric"] = historical_long["metric_code"].map(metric_labels)
historical_long["comparison"] = historical_long["metric_code"].map(
    comparison_labels
)

print("Historical wide shape:", historical_tableau.shape)
print("Historical long shape:", historical_long.shape)

historical_long.head()

Historical wide shape: (356, 14)
Historical long shape: (1780, 13)


,date,year,month_number,month_name,quarter,year_month,economic_period,period_order,volume_growth_status,metric_code,percentage,metric,comparison
0,1997-01-01,1997,1,January,1997Q1,1997-01,Pre-pandemic (1997–2019),1,Strong growth (3% or more),value_yoy_pct,6.2,Sales value growth,Year-on-year
1,1997-02-01,1997,2,February,1997Q1,1997-02,Pre-pandemic (1997–2019),1,Strong growth (3% or more),value_yoy_pct,4.6,Sales value growth,Year-on-year
2,1997-03-01,1997,3,March,1997Q1,1997-03,Pre-pandemic (1997–2019),1,Strong growth (3% or more),value_yoy_pct,5.3,Sales value growth,Year-on-year
3,1997-04-01,1997,4,April,1997Q2,1997-04,Pre-pandemic (1997–2019),1,Strong growth (3% or more),value_yoy_pct,3.9,Sales value growth,Year-on-year
4,1997-05-01,1997,5,May,1997Q2,1997-05,Pre-pandemic (1997–2019),1,Strong growth (3% or more),value_yoy_pct,4.4,Sales value growth,Year-on-year


## 5. Prepare the forecast dataset

Date fields and a historical-versus-forecast classification are added so that the forecast can be displayed clearly in Tableau.

In [7]:
forecast_tableau = (
    forecast
    .sort_values("date")
    .copy()
)

forecast_tableau["year"] = forecast_tableau["date"].dt.year
forecast_tableau["month_number"] = forecast_tableau["date"].dt.month
forecast_tableau["month_name"] = forecast_tableau["date"].dt.month_name()
forecast_tableau["quarter"] = (
    forecast_tableau["date"]
    .dt.to_period("Q")
    .astype(str)
)
forecast_tableau["year_month"] = (
    forecast_tableau["date"]
    .dt.strftime("%Y-%m")
)
forecast_tableau["data_type"] = "Forecast"

forecast_tableau.head()

,date,forecast_volume_yoy_pct,lower_95_pct,upper_95_pct,model,year,month_number,month_name,quarter,year_month,data_type
0,2026-09-01,2.4,-0.544896,5.344896,Naïve,2026,9,September,2026Q3,2026-09,Forecast
1,2026-10-01,2.4,-0.544896,5.344896,Naïve,2026,10,October,2026Q4,2026-10,Forecast
2,2026-11-01,2.4,-0.544896,5.344896,Naïve,2026,11,November,2026Q4,2026-11,Forecast
3,2026-12-01,2.4,-0.544896,5.344896,Naïve,2026,12,December,2026Q4,2026-12,Forecast
4,2027-01-01,2.4,-0.544896,5.344896,Naïve,2027,1,January,2027Q1,2027-01,Forecast


## 6. Combine historical and forecast volume growth

A single dataset is created containing both actual historical observations and future forecasts. This will support a continuous Tableau timeline with separate colours for historical and forecast values.

In [8]:
historical_volume = pd.DataFrame({
    "date": historical_tableau["date"],
    "plot_value_pct": historical_tableau["volume_yoy_pct"],
    "actual_volume_yoy_pct": historical_tableau["volume_yoy_pct"],
    "forecast_volume_yoy_pct": np.nan,
    "lower_95_pct": np.nan,
    "upper_95_pct": np.nan,
    "data_type": "Historical",
    "model": "Observed"
})

forecast_volume = pd.DataFrame({
    "date": forecast_tableau["date"],
    "plot_value_pct": forecast_tableau["forecast_volume_yoy_pct"],
    "actual_volume_yoy_pct": np.nan,
    "forecast_volume_yoy_pct": forecast_tableau[
        "forecast_volume_yoy_pct"
    ],
    "lower_95_pct": forecast_tableau["lower_95_pct"],
    "upper_95_pct": forecast_tableau["upper_95_pct"],
    "data_type": "Forecast",
    "model": forecast_tableau["model"]
})

volume_actual_forecast = (
    pd.concat(
        [historical_volume, forecast_volume],
        ignore_index=True
    )
    .sort_values("date")
    .reset_index(drop=True)
)

volume_actual_forecast["year"] = volume_actual_forecast["date"].dt.year
volume_actual_forecast["month_number"] = (
    volume_actual_forecast["date"].dt.month
)
volume_actual_forecast["month_name"] = (
    volume_actual_forecast["date"].dt.month_name()
)
volume_actual_forecast["year_month"] = (
    volume_actual_forecast["date"].dt.strftime("%Y-%m")
)

transition_view = volume_actual_forecast[
    volume_actual_forecast["date"].between(
        "2026-06-01",
        "2026-11-01"
    )
]

transition_view

,date,plot_value_pct,actual_volume_yoy_pct,forecast_volume_yoy_pct,lower_95_pct,upper_95_pct,data_type,model,year,month_number,month_name,year_month
353,2026-06-01,3.4,3.4,NaN,NaN,NaN,Historical,Observed,2026,6,June,2026-06
354,2026-07-01,1.2,1.2,NaN,NaN,NaN,Historical,Observed,2026,7,July,2026-07
355,2026-08-01,2.4,2.4,NaN,NaN,NaN,Historical,Observed,2026,8,August,2026-08
356,2026-09-01,2.4,NaN,2.4,-0.544896,5.344896,Forecast,Naïve,2026,9,September,2026-09
357,2026-10-01,2.4,NaN,2.4,-0.544896,5.344896,Forecast,Naïve,2026,10,October,2026-10
358,2026-11-01,2.4,NaN,2.4,-0.544896,5.344896,Forecast,Naïve,2026,11,November,2026-11


## 7. Validate the Tableau datasets

Final checks confirm that the prepared datasets contain the expected numbers of rows, valid date ranges and no unintended duplicate records.

In [9]:
assert len(historical_tableau) == len(historical)
assert len(historical_long) == len(historical) * len(metric_columns)
assert len(volume_actual_forecast) == len(historical) + len(forecast)

assert historical_tableau["date"].duplicated().sum() == 0
assert historical_long.duplicated(
    subset=["date", "metric_code"]
).sum() == 0
assert volume_actual_forecast["date"].duplicated().sum() == 0

assert historical_long["percentage"].isna().sum() == 0
assert volume_actual_forecast["plot_value_pct"].isna().sum() == 0

forecast_rows = (
    volume_actual_forecast["data_type"] == "Forecast"
)

assert volume_actual_forecast.loc[
    forecast_rows,
    [
        "forecast_volume_yoy_pct",
        "lower_95_pct",
        "upper_95_pct"
    ]
].notna().all().all()

print("All Tableau dataset validation checks passed.")
print("Historical wide rows:", len(historical_tableau))
print("Historical long rows:", len(historical_long))
print("Actual and forecast rows:", len(volume_actual_forecast))

All Tableau dataset validation checks passed.
Historical wide rows: 356
Historical long rows: 1780
Actual and forecast rows: 368


## 8. Export the Tableau-ready datasets

Three CSV files are exported:

1. A wide historical dataset for detailed calculations.
2. A long historical dataset for flexible metric comparisons.
3. A combined historical-and-forecast dataset for the forecasting visual.

In [10]:
tableau_directory = Path("..") / "tableau"
tableau_directory.mkdir(parents=True, exist_ok=True)

historical_wide_path = (
    tableau_directory
    / "retail_sales_historical_wide.csv"
)

historical_long_path = (
    tableau_directory
    / "retail_sales_historical_long.csv"
)

actual_forecast_path = (
    tableau_directory
    / "retail_sales_actual_forecast.csv"
)

historical_tableau.to_csv(
    historical_wide_path,
    index=False
)

historical_long.to_csv(
    historical_long_path,
    index=False
)

volume_actual_forecast.to_csv(
    actual_forecast_path,
    index=False
)

print("Files exported:")
print(historical_wide_path)
print(historical_long_path)
print(actual_forecast_path)

Files exported:
../tableau/retail_sales_historical_wide.csv
../tableau/retail_sales_historical_long.csv
../tableau/retail_sales_actual_forecast.csv


## Conclusion

The historical retail sales data has been enhanced with date fields, economic-period classifications and volume-growth categories.

A long-format dataset has been created to support flexible comparison of sales value growth, sales volume growth and the implied deflator in Tableau.

Historical sales-volume growth has also been combined with the 12-month forecast, including its approximate uncertainty range. This dataset will support a clear distinction between observed and forecast values.

All prepared datasets passed the validation checks and were exported to the `tableau` folder. They are now ready to be connected to Tableau and used to build the dashboard.